In [33]:
nltk.download ('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\divya\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\divya\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
import pandas as pd

# Load train and test datasets
df_train = pd.read_csv(r"E:\ds\data3\train (5).csv")
df_test  = pd.read_csv(r"E:\ds\data3\test (4).csv")

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)


Train shape: (159571, 8)
Test shape: (153164, 2)


In [3]:
df1= pd.read_csv(r"E:\ds\data3\train (5).csv")
df1

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0
...,...,...,...,...,...,...,...,...
159566,ffe987279560d7ff,""":::::And for the second time of asking, when ...",0,0,0,0,0,0
159567,ffea4adeee384e90,You should be ashamed of yourself \n\nThat is ...,0,0,0,0,0,0
159568,ffee36eab5c267c9,"Spitzer \n\nUmm, theres no actual article for ...",0,0,0,0,0,0
159569,fff125370e4aaaf3,And it looks like it was actually you who put ...,0,0,0,0,0,0


In [4]:
df2  = pd.read_csv(r"E:\ds\data3\test (4).csv")
df2

,id,comment_text
0,00001cee341fdb12,Yo bitch Ja Rule is more succesful then you'll...
1,0000247867823ef7,== From RfC == \n\n The title is fine as it is...
2,00013b17ad220c46,""" \n\n == Sources == \n\n * Zawe Ashton on Lap..."
3,00017563c3f7919a,":If you have a look back at the source, the in..."
4,00017695ad8997eb,I don't anonymously edit articles at all.
...,...,...
153159,fffcd0960ee309b5,". \n i totally agree, this stuff is nothing bu..."
153160,fffd7a9a6eb32c16,== Throw from out field to home plate. == \n\n...
153161,fffda9e8d6fafa9e,""" \n\n == Okinotorishima categories == \n\n I ..."
153162,fffe8f1340a79fc2,""" \n\n == """"One of the founding nations of the..."


In [5]:
print(df1.isnull().any())
print("------------------------------")
print(df1.columns)
print(df1.shape)

id               False
comment_text     False
toxic            False
severe_toxic     False
obscene          False
threat           False
insult           False
identity_hate    False
dtype: bool
------------------------------
Index(['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat',
       'insult', 'identity_hate'],
      dtype='object')
(159571, 8)


In [6]:
print(df2.isnull().any())
print("------------------------------")
print(df2.columns)
print(df2.shape)

id              False
comment_text    False
dtype: bool
------------------------------
Index(['id', 'comment_text'], dtype='object')
(153164, 2)


In [7]:
import re
import nltk
from nltk.corpus import stopwords

# Download stopwords once
nltk.download("stopwords")
stop_words = set(stopwords.words("english"))

# Define the cleaning function
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z]", " ", text)   # remove non-letters
    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words]
    return " ".join(tokens)

def vectorize_text(train_texts, test_texts):
    vectorizer = TfidfVectorizer(max_features=5000)
    X_train = vectorizer.fit_transform(train_texts)
    X_test = vectorizer.transform(test_texts)
    return X_train, X_test, vectorizer

# Apply cleaning on df_train and df_test
df_train["clean_comment"] = df_train["comment_text"].apply(clean_text)
df_test["clean_comment"]  = df_test["comment_text"].apply(clean_text)






[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\divya\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Define the model
class ToxicityModel(nn.Module):
    def __init__(self, input_dim):
        super(ToxicityModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, 256)
        self.dropout1 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(256, 128)
        self.dropout2 = nn.Dropout(0.3)
        self.fc3 = nn.Linear(128, 1)  # Output layer

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)
        x = torch.sigmoid(self.fc3(x))  # Sigmoid for binary classification
        return x


# training the loop

In [9]:
from torch.utils.data import DataLoader, TensorDataset

def train_model(model, train_loader, criterion, optimizer, epochs=5):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs.squeeze(), y_batch.float())
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")


# Evaluation

In [10]:
from sklearn.metrics import classification_report

def evaluate_model(model, X_test, y_test):
    model.eval()
    with torch.no_grad():
        y_prob = model(X_test).squeeze()
        y_pred = (y_prob >= 0.5).int()
    print(classification_report(y_test, y_pred))


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Clean text
df1["clean_comment"] = df1["comment_text"].apply(clean_text)

# Vectorize
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df1["clean_comment"])
y = df1["toxic"]   # or multi-label if you want

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert to tensors
X_train_tensor = torch.tensor(X_train.toarray(), dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test.toarray(), dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32)


In [12]:


# DataLoader
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=32, shuffle=True)

# Build model
model = ToxicityModel(input_dim=X_train_tensor.shape[1])
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Train + Evaluate
train_model(model, train_loader, criterion, optimizer, epochs=5)
evaluate_model(model, X_test_tensor, y_test_tensor)


Epoch 1/5, Loss: 604.0816
Epoch 2/5, Loss: 436.7425
Epoch 3/5, Loss: 348.4812
Epoch 4/5, Loss: 245.4680
Epoch 5/5, Loss: 171.3329
              precision    recall  f1-score   support

         0.0       0.97      0.99      0.98     28859
         1.0       0.85      0.67      0.75      3056

    accuracy                           0.96     31915
   macro avg       0.91      0.83      0.86     31915
weighted avg       0.95      0.96      0.95     31915



#  Model Development

In [13]:
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(LSTMModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = self.embedding(x)
        _, (h, _) = self.lstm(x)
        out = torch.sigmoid(self.fc(h[-1]))
        return out


# Model Saving & Loading

In [14]:
# Save model
torch.save(model.state_dict(), "toxicity_model.pth")

# Load model later
model = ToxicityModel(input_dim=X_train_tensor.shape[1])
model.load_state_dict(torch.load("toxicity_model.pth"))
model.eval()


ToxicityModel(
  (fc1): Linear(in_features=5000, out_features=256, bias=True)
  (dropout1): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (dropout2): Dropout(p=0.3, inplace=False)
  (fc3): Linear(in_features=128, out_features=1, bias=True)
)

In [15]:
import joblib
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")


['tfidf_vectorizer.pkl']

# Experiment with Advanced Architectures

In [20]:
from transformers import BertTokenizer, BertForSequenceClassification
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=6)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

c:\Users\divya\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\divya\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:03<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Model Saving & Loading

In [21]:
torch.save(model.state_dict(), "toxicity_model.pth")
import joblib
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")


['tfidf_vectorizer.pkl']

# Deployment with Streamlit

In [19]:
import streamlit as st

st.title("Toxicity Detector")
user_input = st.text_area("Enter a comment:")

if st.button("Predict"):
    clean = clean_text(user_input)
    vec = vectorizer.transform([clean]).toarray()
    tensor = torch.tensor(vec, dtype=torch.float32)
    prob = model(tensor).detach().numpy()
    st.write("Predictions:", prob)


2026-09-25 02:15:18.488 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-25 02:15:19.510 
  command:

    streamlit run C:\Users\divya\AppData\Roaming\Python\Python314\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-09-25 02:15:19.511 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-25 02:15:19.512 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-25 02:15:19.514 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-25 02:15:19.519 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-25 02:15:19.523 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-25 02:15:19.529 Sess